## Imports

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model
import model_analysis

import importlib
import matplotlib.pyplot as plt
import xarray as xr

import numpy as np

import cartopy.crs as ccrs
from scipy.stats import beta



## Load Data

In [2]:
is_local_data = False
Month_idx = 4
safe = True
start = None
end = None
max_depth = 3

In [3]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

raw_mrsol_for_skew = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=3)


In [4]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=3)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=3)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_skew = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=4)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=4)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_test = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")


some explantions

In [5]:
def shape_target(ds):
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    return ds

In [6]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth").isel(depth=slice(0, max_depth))
    return ds,chunk_mask, detail_mask

In [7]:
# Transform already as one function
#model.transform.Log_Transform_ds()

In [8]:
def shape_input(ds, chunk_mask):
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

In [9]:
#The chunk_mask schould all be the same, this is important that there are no shape problems in the regressions, you can test this with  
#(chunk_mask != chunk_mask_2).sum(),#(chunk_mask != chunk_mask_1).sum(),
#The chunk_mask is used, that the regression can predict a depth.size vector at each gridcell, the masks: detail_mask_mean, detail_mask_var and detail_mask_test are later used to determan which points are real predictions and witch where just placeholders to let the regression run smoothly.

### Linear Regression of the mean

In [10]:
mean_target_noneT = shape_target(raw_mrsol_for_mean)
mean_target_noneT, chunk_mask, detail_mask = mask_stack_target(mean_target_noneT)
mean_target = model.transform.Log_Transform_ds(mean_target_noneT)
mean_target_da = mean_target.mrsol

In [11]:
mean_predictors = shape_input(raw_input_for_mean,chunk_mask)

In [12]:
LinReg_mean = model.stats._parallel_linear_regression.ParLinearRegression()

In [13]:
LinReg_mean.fit(predictors=mean_predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

Hier wäre die meinung einen neuen Run zu verwenden, die frage ist ob sich die variance durch das fehlen des runs zu tief ausfällt. Overfitting korrektur mit 1/(1-param/n_samples)^2? (Später probieren)

In [14]:
var_target_noneT = shape_target(raw_mrsol_for_var)
var_target_noneT, chunk_mask_var, detail_mask_var = mask_stack_target(var_target_noneT)
var_target = model.transform.Log_Transform_ds(var_target_noneT)



In [15]:
var_predictors = shape_input(raw_input_for_var,chunk_mask)

In [16]:
residuals = LinReg_mean.residuals(var_predictors, var_target)

### Linear Regression of the Variance

In [17]:
LinReg_variance = model.stats._parallel_linear_regression.ParLinearRegression()

In [18]:
LinReg_variance.fit(predictors=var_predictors, target=(residuals.residuals)**2,location_dim="gridcell", regr_dim="time")

### skewness 

In [19]:
skew_target_noneT = shape_target(raw_mrsol_for_skew)
skew_target_noneT, chunk_mask_skew, detail_mask_skew = mask_stack_target(skew_target_noneT)
skew_target = model.transform.Log_Transform_ds(skew_target_noneT)


In [20]:
skew_predictors = shape_input(raw_input_for_skew,chunk_mask)

In [21]:
mean_prediction = LinReg_mean.predict(skew_predictors)

In [22]:
residuals = skew_target.mrsol - mean_prediction.prediction

In [23]:
sigmas = np.sqrt(LinReg_variance.predict(skew_predictors).prediction.clip(min = 1e-32))

In [24]:
standardized_values_for_skew = (residuals/sigmas)

### Linear Regression of the Skewness

In [25]:
LinReg_skewness = model.stats._parallel_linear_regression.ParLinearRegression()

In [26]:
LinReg_skewness.fit(predictors=skew_predictors, target=(standardized_values_for_skew)**3,location_dim="gridcell", regr_dim="time")

### Export Prameters

In [27]:
if safe:
    model.save.save_params(LinReg_mean.params,LinReg_variance.params,skew_parameter = LinReg_skewness.params,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/Transformation_Distribution/log_transform/local{is_local_data}/month{Month_idx}", name=f"start={start},end={end}.max_depth{max_depth}")

In [28]:
#model.loading.load_params(var = "mrsol",scen= "historical", folder = "MPI-ESM1-2-LR/variations/Transformation_Distribution", name=f"dist_beta_transform_rel_month_idx= {month_idx}")